# Employee Sentiment Analysis
## A Comprehensive Analysis of Employee Engagement Through Email Sentiment

This notebook performs a complete sentiment analysis on employee email data to:
- Label sentiments, perform EDA, calculate scores, rank employees
- Identify flight risks and build predictive models

**Dataset:** `test (1).xlsx` — 2,191 email records from 10 employees (2010–2011)

In [1]:
# Section 1: Setup & Data Loading
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from datetime import timedelta
import os
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Create visualizations folder
os.makedirs('visualizations', exist_ok=True)

# Load dataset
df = pd.read_excel('test (1).xlsx')
df['date'] = pd.to_datetime(df['date'])
df['employee'] = df['from'].apply(lambda x: x.split('@')[0].replace('.', ' ').title())

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDate Range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nUnique Employees: {df['employee'].nunique()}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nFirst 5 rows:")
df.head()

Dataset Shape: (2191, 5)

Columns: ['Subject', 'body', 'date', 'from', 'employee']

Date Range: 2010-01-01 to 2011-12-31

Unique Employees: 10

Missing Values:
Subject     0
body        0
date        0
from        0
employee    0
dtype: int64

First 5 rows:


,Subject,body,date,from,employee
0,EnronOptions Update!,EnronOptions Announcement\n\n\nWe have updated...,2010-05-10,sally.beck@enron.com,Sally Beck
1,(No Subject),"Marc,\n\nUnfortunately, today is not going to ...",2010-07-29,eric.bass@enron.com,Eric Bass
2,Phone Screen Interview - Shannon L. Burnham,"When: Wednesday, June 06, 2001 10:00 AM-11:00 ...",2011-07-25,sally.beck@enron.com,Sally Beck
3,RE: My new work email,we were thinking papasitos (we can meet somewh...,2010-03-25,johnny.palmer@enron.com,Johnny Palmer
4,Bet,Since you never gave me the $20 for the last t...,2011-05-21,lydia.delgado@enron.com,Lydia Delgado


---
## Task 1: Sentiment Labeling

**Approach:** We use **TextBlob**, a Python NLP library, to compute the polarity of each email body.

TextBlob returns a polarity score between -1 (most negative) and +1 (most positive).

**Classification Thresholds:**
- **Positive:** polarity > 0.05
- **Negative:** polarity < -0.05
- **Neutral:** -0.05 ≤ polarity ≤ 0.05

These thresholds avoid misclassifying near-zero polarity texts.

In [2]:
# Task 1: Sentiment Labeling using TextBlob
def get_sentiment(text):
    if pd.isna(text) or str(text).strip() == '':
        return 'Neutral'
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0.05:
        return 'Positive'
    elif polarity < -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Apply sentiment labeling
df['Sentiment'] = df['body'].apply(get_sentiment)
df['polarity'] = df['body'].apply(lambda x: TextBlob(str(x)).sentiment.polarity if pd.notna(x) else 0)

print("Sentiment Distribution:")
print(df['Sentiment'].value_counts())
print(f"\nSentiment Percentages:")
print((df['Sentiment'].value_counts(normalize=True) * 100).round(2))

# Show sample labeled messages
print("\n--- Sample Labeled Messages ---")
for sent in ['Positive', 'Negative', 'Neutral']:
    sample = df[df['Sentiment'] == sent].iloc[0]
    print(f"\n[{sent}] (polarity={sample['polarity']:.3f})")
    print(f"  Subject: {sample['Subject'][:80]}")
    print(f"  Body: {str(sample['body'])[:120]}...")

Sentiment Distribution:
Sentiment
Positive    1144
Neutral      817
Negative     230
Name: count, dtype: int64

Sentiment Percentages:
Sentiment
Positive    52.21
Neutral     37.29
Negative    10.50
Name: proportion, dtype: float64

--- Sample Labeled Messages ---

[Positive] (polarity=0.250)
  Subject: EnronOptions Update!
  Body: EnronOptions Announcement


We have updated the EnronOptions =01) Your Stock Option Program web site!  =
The=20
web site...

[Negative] (polarity=-0.075)
  Subject: (No Subject)
  Body: Marc,

Unfortunately, today is not going to work for the revenue model for mid & 
back office services meeting.  How abo...

[Neutral] (polarity=0.000)
  Subject: Phone Screen  Interview - Shannon L. Burnham
  Body: When: Wednesday, June 06, 2001 10:00 AM-11:00 AM (GMT-06:00) Central Time (US & Canada).
Where:  @ 10:00am CST  (225) 93...


---
## Task 2: Exploratory Data Analysis (EDA)

In this section, we explore the dataset structure, sentiment distributions, temporal trends, and per-employee patterns to build a foundation for subsequent analysis.

In [3]:
# 2.1 Dataset Overview
print("=== Dataset Structure ===")
print(f"Total Records: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nBasic Statistics:")
df.describe(include='all')

=== Dataset Structure ===
Total Records: 2191
Columns: ['Subject', 'body', 'date', 'from', 'employee', 'Sentiment', 'polarity']

Data Types:
Subject              object
body                 object
date         datetime64[ns]
from                 object
employee             object
Sentiment            object
polarity            float64
dtype: object

Basic Statistics:


,Subject,body,date,from,employee,Sentiment,polarity
count,2191,2191,2191,2191,2191,2191,2191.000000
unique,1251,1539,NaN,10,10,3,NaN
top,(No Subject),\n\n,NaN,lydia.delgado@enron.com,Lydia Delgado,Positive,NaN
freq,141,21,NaN,284,284,1144,NaN
mean,NaN,NaN,2010-12-31 02:17:21.716111360,NaN,NaN,NaN,0.119929
min,NaN,NaN,2010-01-01 00:00:00,NaN,NaN,NaN,-1.000000
25%,NaN,NaN,2010-06-30 12:00:00,NaN,NaN,NaN,0.000000
50%,NaN,NaN,2011-01-01 00:00:00,NaN,NaN,NaN,0.070833
75%,NaN,NaN,2011-06-30 12:00:00,NaN,NaN,NaN,0.204383
max,NaN,NaN,2011-12-31 00:00:00,NaN,NaN,NaN,1.000000


In [4]:
# 2.2 Sentiment Distribution — Bar Chart & Pie Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
colors = {'Positive': '#2ecc71', 'Neutral': '#3498db', 'Negative': '#e74c3c'}
sent_counts = df['Sentiment'].value_counts()
sent_counts.plot(kind='bar', ax=axes[0], color=[colors[x] for x in sent_counts.index], edgecolor='black')
axes[0].set_title('Sentiment Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(sent_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
sent_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90,
                 colors=[colors[x] for x in sent_counts.index])
axes[1].set_title('Sentiment Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('visualizations/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/sentiment_distribution.png")

Saved: visualizations/sentiment_distribution.png


In [5]:
# 2.3 Sentiment Trends Over Time (Monthly)
df['year_month'] = df['date'].dt.to_period('M')
monthly_sentiment = df.groupby(['year_month', 'Sentiment']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(16, 6))
monthly_sentiment.plot(kind='line', ax=ax, marker='o',
                       color=[colors.get(c, '#333') for c in monthly_sentiment.columns], linewidth=2)
ax.set_title('Monthly Sentiment Trends Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Messages')
ax.legend(title='Sentiment')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/sentiment_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/sentiment_over_time.png")

Saved: visualizations/sentiment_over_time.png


In [6]:
# 2.4 Per-Employee Sentiment Breakdown (Stacked Bar)
emp_sent = df.groupby(['employee', 'Sentiment']).size().unstack(fill_value=0)
emp_sent = emp_sent.reindex(columns=['Positive', 'Neutral', 'Negative'], fill_value=0)

fig, ax = plt.subplots(figsize=(14, 7))
emp_sent.plot(kind='barh', stacked=True, ax=ax,
              color=['#2ecc71', '#3498db', '#e74c3c'], edgecolor='black')
ax.set_title('Sentiment Breakdown by Employee', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Messages')
ax.set_ylabel('Employee')
ax.legend(title='Sentiment')
plt.tight_layout()
plt.savefig('visualizations/employee_sentiment_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/employee_sentiment_breakdown.png")

Saved: visualizations/employee_sentiment_breakdown.png


In [7]:
# 2.5 Message Frequency Heatmap by Month and Employee
df['month_str'] = df['date'].dt.strftime('%Y-%m')
heatmap_data = df.pivot_table(index='employee', columns='month_str', aggfunc='size', fill_value=0)

fig, ax = plt.subplots(figsize=(20, 8))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, linewidths=0.5, ax=ax)
ax.set_title('Message Frequency Heatmap (Employee × Month)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Employee')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('visualizations/message_frequency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/message_frequency_heatmap.png")

Saved: visualizations/message_frequency_heatmap.png


In [8]:
# 2.6 Message Length Analysis
df['msg_length'] = df['body'].astype(str).apply(len)
df['word_count'] = df['body'].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df.boxplot(column='msg_length', by='Sentiment', ax=axes[0])
axes[0].set_title('Message Length by Sentiment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Character Count')
plt.sca(axes[0])
plt.xticks(rotation=0)

df.boxplot(column='word_count', by='Sentiment', ax=axes[1])
axes[1].set_title('Word Count by Sentiment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word Count')
plt.sca(axes[1])
plt.xticks(rotation=0)

plt.suptitle('')
plt.tight_layout()
plt.savefig('visualizations/message_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/message_length_analysis.png")

Saved: visualizations/message_length_analysis.png


In [9]:
# 2.7 Polarity Distribution Histogram
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df['polarity'], bins=50, color='#8e44ad', edgecolor='black', alpha=0.7)
ax.axvline(x=0.05, color='green', linestyle='--', label='Positive threshold (0.05)')
ax.axvline(x=-0.05, color='red', linestyle='--', label='Negative threshold (-0.05)')
ax.set_title('Distribution of Polarity Scores', fontsize=14, fontweight='bold')
ax.set_xlabel('Polarity')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('visualizations/polarity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/polarity_distribution.png")

Saved: visualizations/polarity_distribution.png


### EDA Observations
Key findings from the exploratory analysis:
1. **Sentiment Split:** The majority of messages tend to be Positive or Neutral, with a smaller proportion being Negative.
2. **Temporal Trends:** Sentiment patterns fluctuate month to month, reflecting varying engagement levels.
3. **Employee Differences:** Each employee shows a distinct sentiment profile — some are consistently positive, others show more negativity.
4. **Message Length:** There is no strong correlation between message length and sentiment polarity.

---
## Task 3: Employee Score Calculation

**Scoring Method:**
- Positive message: **+1**
- Negative message: **−1**
- Neutral message: **0**

Scores are aggregated monthly for each employee and reset at the start of each new month.

In [10]:
# Task 3: Monthly Employee Score Calculation
df['score'] = df['Sentiment'].map({'Positive': 1, 'Negative': -1, 'Neutral': 0})

monthly_scores = df.groupby(['employee', 'year_month']).agg(
    total_messages=('score', 'count'),
    positive_count=('score', lambda x: (x == 1).sum()),
    negative_count=('score', lambda x: (x == -1).sum()),
    neutral_count=('score', lambda x: (x == 0).sum()),
    sentiment_score=('score', 'sum')
).reset_index()

monthly_scores['year_month_str'] = monthly_scores['year_month'].astype(str)

print("=== Monthly Employee Sentiment Scores ===")
print(monthly_scores.to_string(index=False))

=== Monthly Employee Sentiment Scores ===
      employee year_month  total_messages  positive_count  negative_count  neutral_count  sentiment_score year_month_str
 Bobette Riner    2010-01               2               2               0              0                2        2010-01
 Bobette Riner    2010-02              14               9               1              4                8        2010-02
 Bobette Riner    2010-03              11               4               0              7                4        2010-03
 Bobette Riner    2010-04               6               4               0              2                4        2010-04
 Bobette Riner    2010-05               4               1               0              3                1        2010-05
 Bobette Riner    2010-06               5               1               1              3                0        2010-06
 Bobette Riner    2010-07              12               9               1              2                8      

In [11]:
# Visualize monthly scores for all employees
fig, ax = plt.subplots(figsize=(16, 8))
for emp in sorted(monthly_scores['employee'].unique()):
    emp_data = monthly_scores[monthly_scores['employee'] == emp]
    ax.plot(emp_data['year_month_str'], emp_data['sentiment_score'],
            marker='o', label=emp, linewidth=2)

ax.set_title('Monthly Sentiment Scores by Employee', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Sentiment Score')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('visualizations/employee_monthly_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/employee_monthly_scores.png")

Saved: visualizations/employee_monthly_scores.png


---
## Task 4: Employee Ranking

For each month, we identify:
- **Top 3 Positive Employees:** Highest sentiment scores (descending score, then alphabetical)
- **Top 3 Negative Employees:** Lowest sentiment scores (ascending score, then alphabetical)

In [12]:
# Task 4: Employee Ranking by Month
print("=" * 80)
print("MONTHLY EMPLOYEE RANKINGS")
print("=" * 80)

all_top_positive = []
all_top_negative = []

for month in sorted(monthly_scores['year_month'].unique()):
    month_data = monthly_scores[monthly_scores['year_month'] == month].copy()

    # Top 3 Positive: sort by score DESC, then employee name ASC
    top_pos = month_data.sort_values(['sentiment_score', 'employee'],
                                      ascending=[False, True]).head(3)
    # Top 3 Negative: sort by score ASC, then employee name ASC
    top_neg = month_data.sort_values(['sentiment_score', 'employee'],
                                      ascending=[True, True]).head(3)

    all_top_positive.append(top_pos.assign(month=str(month)))
    all_top_negative.append(top_neg.assign(month=str(month)))

    print(f"\n--- {month} ---")
    print(f"  Top 3 Positive:")
    for i, (_, row) in enumerate(top_pos.iterrows(), 1):
        print(f"    {i}. {row['employee']} (Score: {row['sentiment_score']})")
    print(f"  Top 3 Negative:")
    for i, (_, row) in enumerate(top_neg.iterrows(), 1):
        print(f"    {i}. {row['employee']} (Score: {row['sentiment_score']})")

top_pos_df = pd.concat(all_top_positive, ignore_index=True)
top_neg_df = pd.concat(all_top_negative, ignore_index=True)

MONTHLY EMPLOYEE RANKINGS

--- 2010-01 ---
  Top 3 Positive:
    1. Kayne Coulter (Score: 8)
    2. Don Baughman (Score: 5)
    3. Eric Bass (Score: 5)
  Top 3 Negative:
    1. Rhonda Denton (Score: 0)
    2. Johnny Palmer (Score: 1)
    3. Bobette Riner (Score: 2)

--- 2010-02 ---
  Top 3 Positive:
    1. Bobette Riner (Score: 8)
    2. John Arnold (Score: 8)
    3. Don Baughman (Score: 6)
  Top 3 Negative:
    1. Lydia Delgado (Score: 1)
    2. Patti Thompson (Score: 1)
    3. Sally Beck (Score: 1)

--- 2010-03 ---
  Top 3 Positive:
    1. Sally Beck (Score: 7)
    2. Lydia Delgado (Score: 6)
    3. Bobette Riner (Score: 4)
  Top 3 Negative:
    1. Eric Bass (Score: 0)
    2. Rhonda Denton (Score: 0)
    3. Don Baughman (Score: 2)

--- 2010-04 ---
  Top 3 Positive:
    1. Don Baughman (Score: 8)
    2. John Arnold (Score: 5)
    3. Bobette Riner (Score: 4)
  Top 3 Negative:
    1. Eric Bass (Score: 1)
    2. Patti Thompson (Score: 2)
    3. Rhonda Denton (Score: 3)

--- 2010-05 ---
 

In [13]:
# Visualize top positive and negative employees (overall frequency in top 3)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pos_freq = top_pos_df['employee'].value_counts()
pos_freq.plot(kind='bar', ax=axes[0], color='#2ecc71', edgecolor='black')
axes[0].set_title('Times in Top 3 Positive', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Frequency')
axes[0].tick_params(axis='x', rotation=45)

neg_freq = top_neg_df['employee'].value_counts()
neg_freq.plot(kind='bar', ax=axes[1], color='#e74c3c', edgecolor='black')
axes[1].set_title('Times in Top 3 Negative', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Frequency')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('visualizations/employee_rankings.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/employee_rankings.png")

Saved: visualizations/employee_rankings.png


---
## Task 5: Flight Risk Identification

**Definition:** A flight risk is any employee who has sent **4 or more negative emails** within any **rolling 30-day window**.

**Approach:** For each employee, we iterate through their negative messages sorted by date and use a sliding window to check if any 30-day period contains 4+ negative messages.

In [14]:
# Task 5: Flight Risk Identification (Rolling 30-day window)
flight_risks = []

negative_msgs = df[df['Sentiment'] == 'Negative'].sort_values('date')

for emp in negative_msgs['from'].unique():
    emp_neg = negative_msgs[negative_msgs['from'] == emp].sort_values('date')
    dates = emp_neg['date'].tolist()
    emp_name = emp_neg['employee'].iloc[0]

    for i in range(len(dates)):
        window_end = dates[i] + timedelta(days=30)
        count = sum(1 for d in dates if dates[i] <= d <= window_end)
        if count >= 4:
            flight_risks.append({
                'employee': emp_name,
                'email': emp,
                'window_start': dates[i].date(),
                'window_end': window_end.date(),
                'negative_count_in_window': count
            })
            break  # Flag employee once

flight_risk_df = pd.DataFrame(flight_risks)

print("=" * 60)
print("FLIGHT RISK EMPLOYEES")
print("=" * 60)
if len(flight_risk_df) > 0:
    for _, row in flight_risk_df.iterrows():
        print(f"  ⚠ {row['employee']} ({row['email']})")
        print(f"    Window: {row['window_start']} to {row['window_end']}")
        print(f"    Negative messages in window: {row['negative_count_in_window']}")
        print()
else:
    print("  No flight risk employees identified.")

print(f"\nTotal flight risk employees: {len(flight_risk_df)}")

FLIGHT RISK EMPLOYEES
  ⚠ Lydia Delgado (lydia.delgado@enron.com)
    Window: 2011-05-04 to 2011-06-03
    Negative messages in window: 4

  ⚠ Johnny Palmer (johnny.palmer@enron.com)
    Window: 2010-02-14 to 2010-03-16
    Negative messages in window: 5

  ⚠ John Arnold (john.arnold@enron.com)
    Window: 2010-05-25 to 2010-06-24
    Negative messages in window: 5

  ⚠ Eric Bass (eric.bass@enron.com)
    Window: 2010-09-13 to 2010-10-13
    Negative messages in window: 5

  ⚠ Patti Thompson (patti.thompson@enron.com)
    Window: 2011-03-06 to 2011-04-05
    Negative messages in window: 4

  ⚠ Sally Beck (sally.beck@enron.com)
    Window: 2010-06-16 to 2010-07-16
    Negative messages in window: 5

  ⚠ Bobette Riner (bobette.riner@ipgdirect.com)
    Window: 2010-11-02 to 2010-12-02
    Negative messages in window: 5

  ⚠ Rhonda Denton (rhonda.denton@enron.com)
    Window: 2010-12-18 to 2011-01-17
    Negative messages in window: 4


Total flight risk employees: 8


In [15]:
# Flight Risk Visualization
if len(flight_risk_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(flight_risk_df['employee'], flight_risk_df['negative_count_in_window'],
            color='#e74c3c', edgecolor='black')
    ax.set_title('Flight Risk Employees — Negative Messages in 30-Day Window',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Negative Message Count')
    ax.axvline(x=4, color='black', linestyle='--', alpha=0.5, label='Threshold (4)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('visualizations/flight_risk_employees.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: visualizations/flight_risk_employees.png")
else:
    print("No flight risk chart to generate.")

Saved: visualizations/flight_risk_employees.png


---
## Task 6: Predictive Modeling (Linear Regression)

**Objective:** Build a linear regression model to predict monthly sentiment scores using engineered features.

**Features selected:**
- `total_messages`: Total messages sent in the month
- `avg_msg_length`: Average message length (characters)
- `avg_word_count`: Average word count per message
- `positive_ratio`: Proportion of positive messages
- `negative_ratio`: Proportion of negative messages

In [16]:
# Task 6: Feature Engineering for Predictive Modeling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Aggregate features per employee per month
df['year_month'] = df['date'].dt.to_period('M')

features_df = df.groupby(['employee', 'year_month']).agg(
    total_messages=('body', 'count'),
    avg_msg_length=('msg_length', 'mean'),
    avg_word_count=('word_count', 'mean'),
    sentiment_score=('score', 'sum'),
    positive_count=('score', lambda x: (x == 1).sum()),
    negative_count=('score', lambda x: (x == -1).sum()),
).reset_index()

features_df['positive_ratio'] = features_df['positive_count'] / features_df['total_messages']
features_df['negative_ratio'] = features_df['negative_count'] / features_df['total_messages']

feature_cols = ['total_messages', 'avg_msg_length', 'avg_word_count',
                'positive_ratio', 'negative_ratio']
X = features_df[feature_cols]
y = features_df['sentiment_score']

print(f"Feature matrix shape: {X.shape}")
print(f"\nFeature Statistics:\n{X.describe().round(2)}")
print(f"\nCorrelation with target (sentiment_score):")
for col in feature_cols:
    print(f"  {col}: {features_df[col].corr(y):.4f}")

Feature matrix shape: (240, 5)

Feature Statistics:
       total_messages  avg_msg_length  avg_word_count  positive_ratio  \
count          240.00          240.00          240.00          240.00   
mean             9.13          254.75           40.96            0.54   
std              5.73          104.00           17.57            0.24   
min              1.00            3.00            0.00            0.00   
25%              4.75          196.64           30.66            0.40   
50%              9.00          244.38           39.87            0.53   
75%             12.00          299.09           48.79            0.67   
max             27.00          725.00          126.00            1.00   

       negative_ratio  
count          240.00  
mean             0.10  
std              0.12  
min              0.00  
25%              0.00  
50%              0.07  
75%              0.16  
max              1.00  

Correlation with target (sentiment_score):
  total_messages: 0.7441
  avg

In [17]:
# Train/Test Split and Model Training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Model Evaluation
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("=== Linear Regression Model Results ===")
print(f"  R² Score:  {r2:.4f}")
print(f"  MSE:       {mse:.4f}")
print(f"  RMSE:      {rmse:.4f}")
print(f"  MAE:       {mae:.4f}")

print(f"\n=== Feature Coefficients ===")
for feat, coef in zip(feature_cols, model.coef_):
    print(f"  {feat:20s}: {coef:+.4f}")
print(f"  {'Intercept':20s}: {model.intercept_:+.4f}")

=== Linear Regression Model Results ===
  R² Score:  0.7606
  MSE:       2.0925
  RMSE:      1.4466
  MAE:       1.0300

=== Feature Coefficients ===
  total_messages      : +0.4176
  avg_msg_length      : +0.0126
  avg_word_count      : -0.0638
  positive_ratio      : +3.9097
  negative_ratio      : -6.4986
  Intercept           : -2.1305


In [18]:
# Model Performance Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.6, color='#3498db', edgecolor='black')
min_val = min(y_test.min(), y_pred.min()) - 1
max_val = max(y_test.max(), y_pred.max()) + 1
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_title('Actual vs Predicted Sentiment Scores', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Score')
axes[0].set_ylabel('Predicted Score')
axes[0].legend()
axes[0].text(0.05, 0.95, f'R² = {r2:.4f}', transform=axes[0].transAxes,
             fontsize=12, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Residual Plot
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, color='#e74c3c', edgecolor='black')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1].set_title('Residual Plot', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.savefig('visualizations/model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/model_performance.png")

Saved: visualizations/model_performance.png


In [19]:
# Feature Importance Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=True)
colors_coef = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_coef, edgecolor='black')
ax.set_title('Feature Coefficients (Linear Regression)', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: visualizations/feature_importance.png")

Saved: visualizations/feature_importance.png


### Model Interpretation
The linear regression model reveals which features most influence monthly sentiment scores:
- **Positive/Negative ratios** have the strongest impact, as expected
- **Message frequency and length** show secondary effects
- The R² score indicates how well the model captures sentiment variance

---
## Summary & Conclusions

### Key Findings:
1. **Sentiment Distribution:** The dataset shows a mix of positive, neutral, and negative sentiments across all employees.
2. **Employee Scores:** Monthly sentiment scores vary significantly across employees and over time.
3. **Rankings:** Consistent patterns emerge in which employees tend to appear in top positive vs negative lists.
4. **Flight Risks:** Employees flagged as flight risks show concentrated bursts of negative messaging.
5. **Predictive Model:** Linear regression provides insights into features that drive sentiment scores.

### Recommendations:
- Regularly monitor employees with declining sentiment trends
- Investigate root causes for employees flagged as flight risks
- Use sentiment analysis as one input in broader employee engagement strategies

In [20]:
# Final Summary Output
print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)

# Overall top 3 positive (by total score across all months)
overall_scores = monthly_scores.groupby('employee')['sentiment_score'].sum().reset_index()

top3_pos = overall_scores.sort_values(['sentiment_score', 'employee'],
                                       ascending=[False, True]).head(3)
print("\nTop 3 Most Positive Employees (Overall):")
for i, (_, r) in enumerate(top3_pos.iterrows(), 1):
    print(f"  {i}. {r['employee']} (Total Score: {r['sentiment_score']})")

top3_neg = overall_scores.sort_values(['sentiment_score', 'employee'],
                                       ascending=[True, True]).head(3)
print("\nTop 3 Most Negative Employees (Overall):")
for i, (_, r) in enumerate(top3_neg.iterrows(), 1):
    print(f"  {i}. {r['employee']} (Total Score: {r['sentiment_score']})")

print("\nFlight Risk Employees:")
if len(flight_risk_df) > 0:
    for _, r in flight_risk_df.iterrows():
        print(f"  ⚠ {r['employee']}")
else:
    print("  None identified")

print("\n" + "=" * 60)
print("All visualizations saved in 'visualizations/' folder.")
print("=" * 60)

PROJECT SUMMARY

Top 3 Most Positive Employees (Overall):
  1. Lydia Delgado (Total Score: 123)
  2. John Arnold (Total Score: 110)
  3. Sally Beck (Total Score: 96)

Top 3 Most Negative Employees (Overall):
  1. Rhonda Denton (Total Score: 59)
  2. Kayne Coulter (Total Score: 75)
  3. Bobette Riner (Total Score: 88)

Flight Risk Employees:
  ⚠ Lydia Delgado
  ⚠ Johnny Palmer
  ⚠ John Arnold
  ⚠ Eric Bass
  ⚠ Patti Thompson
  ⚠ Sally Beck
  ⚠ Bobette Riner
  ⚠ Rhonda Denton

All visualizations saved in 'visualizations/' folder.
